In [ ]:
import json
from eval.eval_moledit import eval_moledit_from_list
from eval.utils import tranform_str_to_json
print("success")

success


In [2]:
def evaluate_moledit_score(model_name): 
    result_dict = dict()
    
    for task in ['add', 'delete', 'sub']:
        print(model_name, task)
        file_name = f"logs/{task}/{model_name}.json" 
        pred_results = json.load(open(file_name, "r"))
        
        invalid_number = 0
        pred_list, src_list = list(), list()
        group_a, group_b = list(), list()
        
        for pred in pred_results:
            ## extract predicted-smiles, if prediction is not json format, need to change 
            if type(pred['json_results']) is str:
                pred_json = tranform_str_to_json(str_input=pred['json_results'])
                # if model_name == 'gemini': pred_json = pred_json[0]
                if pred_json == None:
                    invalid_number += 1
                    continue
                else:
                    if 'output' not in pred_json.keys():
                        invalid_number += 1; continue
                    pred_list.append(pred_json['output'])
                    src_list.append(pred['molecule'])
                    if task == 'add': group_a.append(pred['added_group'])
                    elif task == 'delete': group_a.append(pred['removed_group'])
                    elif task == 'sub':
                        group_a.append(pred['added_group']); group_b.append(pred['removed_group'])
            else:
                pred_list.append(pred['json_results']['output'])
                src_list.append(pred['molecule'])
                if task == 'add': group_a.append(pred['added_group'])
                elif task == 'delete': group_a.append(pred['removed_group'])
                elif task == 'sub':
                    group_a.append(pred['added_group']); group_b.append(pred['removed_group'])
        
        assert len(src_list) == len(pred_list)
        assert len(src_list) == len(group_a)
        
        result_dict[task] = eval_moledit_from_list(src_list=src_list, pred_list=pred_list, group_a=group_a, group_b=group_b, task=task, total_number=len(pred_list)) 
    
    print(f"eval_score_{model_name}", result_dict)
    # json.dump(result_dict, open(f"logs/eval_score_{model_name}.json", "w"), indent=4)


if __name__ == "__main__":
    model_list = ["qwen3-8b"]
    for model_name in model_list:
        evaluate_moledit_score(model_name=model_name)

qwen3-8b add
无效的目标分子SMILES: O=S(=O)(Cc1nc(-c2cccs2)no1)c1ccc2ccccc2n1CHO
无效的目标分子SMILES: COc1cccc(C(=O)N[C@@H](c2ccco2)c2c(O)ccc3ccccc23)c1=O
添加amide失败: 目标分子中amide数量为0, 源分子中amide数量为0
添加amide失败: 目标分子中amide数量为1, 源分子中amide数量为1
添加amine失败: 目标分子中amine数量为0, 源分子中amine数量为0
添加amine失败: 目标分子中amine数量为0, 源分子中amine数量为0
添加benzene_ring失败: 目标分子中benzene_ring数量为1, 源分子中benzene_ring数量为1
添加carboxyl失败: 目标分子中carboxyl数量为0, 源分子中carboxyl数量为0
无效的目标分子SMILES: Cc1ccc(NC(=O)c2ccc(NC(=O)c3ccco3)cc2)cc1[N+](=O)[O-]COOH
添加nitrile失败: 目标分子中nitrile数量为0, 源分子中nitrile数量为0
添加nitrile失败: 目标分子中nitrile数量为0, 源分子中nitrile数量为0
无效的目标分子SMILES: Cc1ccc(-c2ccc(C(=O)NC(=S)Nc3ccc(S(N)(=O)=O)cc3)no2)cc1Cl
无效的目标分子SMILES: Cc1ccccc1OCC(=O)NCCC(=O)[O-]S
qwen3-8b delete
无效的目标分子SMILES: CCN(Cc1ccccn1)c1ccc1o1
无效的目标分子SMILES: CCSCCOC(=O)/C=C/c1cccc1
qwen3-8b sub
无效的目标分子SMILES: Cc1cc(C)nc(C)n1-c1ccc(OCc2ccccc2)cc1O
无效的目标分子SMILES: Cc1ccc(OCCOc2ccccc2)ncn1
无效的目标分子SMILES: CC(=O)Nc1ccc(NC(=O)c2oc(COc3cccClcc3)cc2C)cc1
无效的目标分子SMILES: COc1ccc(N2CCN(S(=O)(=O)c3

[01:55:49] SMILES Parse Error: syntax error while parsing: O=S(=O)(Cc1nc(-c2cccs2)no1)c1ccc2ccccc2n1CHO
[01:55:49] SMILES Parse Error: Failed parsing SMILES 'O=S(=O)(Cc1nc(-c2cccs2)no1)c1ccc2ccccc2n1CHO' for input: 'O=S(=O)(Cc1nc(-c2cccs2)no1)c1ccc2ccccc2n1CHO'
[01:55:49] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[01:55:49] SMILES Parse Error: syntax error while parsing: Cc1ccc(NC(=O)c2ccc(NC(=O)c3ccco3)cc2)cc1[N+](=O)[O-]COOH
[01:55:49] SMILES Parse Error: Failed parsing SMILES 'Cc1ccc(NC(=O)c2ccc(NC(=O)c3ccco3)cc2)cc1[N+](=O)[O-]COOH' for input: 'Cc1ccc(NC(=O)c2ccc(NC(=O)c3ccco3)cc2)cc1[N+](=O)[O-]COOH'
[01:55:49] Can't kekulize mol.  Unkekulized atoms: 5 6 7 8 25
[01:55:49] Explicit valence for atom # 16 O, 3, is greater than permitted
[01:55:49] SMILES Parse Error: unclosed ring for input: 'CCN(Cc1ccccn1)c1ccc1o1'
[01:55:49] Can't kekulize mol.  Unkekulized atoms: 10 11 12 13 14
[01:55:49] Can't kekulize mol.  Unkekulized atoms: 1 2 3 5 6
[01:55:49] Can't kekulize mol.  Unk